# Day 10 — Async Python
### Vidya Sankalp | Python for AI Engineering

**Goal:** Learn how Python can handle multiple slow I/O operations efficiently.

By the end of this notebook, you will understand:
- Why sequential waiting can be slow
- `async def`, `await`, and `asyncio.run()`
- Running independent tasks concurrently with `asyncio.gather()`
- Handling failures and timeouts
- Calling APIs asynchronously with HTTPX
- Validating API data with Pydantic

> We will start with simple examples that do not need an API key or database. The external API section requires an internet connection.

## 1. Why do we need async?

Imagine that an application needs information about three customers. Each request takes one second.

In a sequential program, we wait for customer 1, then customer 2, then customer 3. If each takes one second, the total waiting time is roughly three seconds.

With asynchronous programming, we can start all three independent requests and let their waiting time overlap.

**Important:** Async is especially useful for I/O-bound work—such as waiting for APIs, databases, or files. It does not automatically speed up CPU-heavy calculations.

In [1]:
import time

def get_customer(customer_id):
    # Simulate waiting for an external service
    time.sleep(1)
    return f"Customer {customer_id}"

start = time.perf_counter()

customers = [
    get_customer(1),
    get_customer(2),
    get_customer(3),
]

end = time.perf_counter()

print(customers)
print(f"Sequential time: { end - start:.2f} seconds")

['Customer 1', 'Customer 2', 'Customer 3']
Sequential time: 3.01 seconds


The example above blocks while each call waits. Now let's use `asyncio.sleep()` to simulate the same waiting without blocking the event loop.

## 2. Three important building blocks

| Syntax | Plain-English meaning |
|---|---|
| `async def` | Defines an asynchronous function (a coroutine function). |
| `await` | Waits for an asynchronous operation to finish while allowing other tasks to make progress. |
| `asyncio.run()` | Starts and manages an event loop for a simple async program. |

Think of `await` as: **“I need this result, but while I'm waiting, other async work can progress.”**

In [11]:
import asyncio

async def get_customer_async(customer_id):
    await asyncio.sleep(1)  # Simulate an async network wait
    return f"Customer {customer_id}"

async def main():
    customer = await get_customer_async(1)
    customer2 = await get_customer_async(2)
    customer3 = await get_customer_async(3)
    print(customer)
    print(customer2)
    print(customer3)
start = time.perf_counter()
await main()
end = time.perf_counter()
print(f"Sequential time: { end - start:.2f} seconds")

Customer 1
Customer 2
Customer 3
Sequential time: 3.00 seconds


In [ ]:
import asyncio

async def get_customer_async(customer_id):
    task = asyncio.current_task()
    print(f"Started: {task.get_name()}")

    await asyncio.sleep(10)

    print(f"Finished: {task.get_name()}")
    return f"Customer {customer_id}"

async def main():
    task1 = asyncio.create_task(get_customer_async(1), name="customer-1")
    task2 = asyncio.create_task(get_customer_async(2), name="customer-2")
    task3 = asyncio.create_task(get_customer_async(3), name="customer-3")

    customer1 = await task1
    customer2 = await task2
    customer3 = await task3

    print(customer1, customer2, customer3)

await main()  # In a notebook

Started: customer-1
Started: customer-2
Started: customer-3


## 3. Run independent tasks concurrently with `asyncio.gather()`

`asyncio.gather()` lets us wait for multiple async operations together.

Use it when the tasks are independent—for example, fetching customer, order, and product data from separate services.

In [4]:
import asyncio
import time

async def get_customer_async(customer_id):
    await asyncio.sleep(1)  # Simulate an async network wait
    return f"Customer {customer_id}"

async def main():
    start = time.perf_counter()

    results = await asyncio.gather(
        get_customer_async(1),
        get_customer_async(2),
        get_customer_async(3),
    )

    print(results)
    print(f"Concurrent time: {time.perf_counter() - start:.2f} seconds")

await main()

['Customer 1', 'Customer 2', 'Customer 3']
Concurrent time: 1.00 seconds


**What to notice:** Each task still waits for one second, but the waits overlap. The total time should be around one second rather than three (small differences are normal).

`gather()` is not magic: if tasks depend on one another, you may need to run them in sequence.

## 4. Handle errors and timeouts

Real services can fail. A request may time out, return an error, or be unavailable.

First, let's see how one async task can raise an exception.

In [11]:
async def get_order(order_id, time_to_sleep):
    await asyncio.sleep(time_to_sleep)

    if order_id == 2:
        raise ValueError("Order 2 was not found")

    return {"order_id": order_id, "status": "shipped"}

async def main():
    try:
        task1 = asyncio.create_task(get_order(1, 20))
        task2 = asyncio.create_task(get_order(2,0.2))
        order1 = await task1
        order2 = await task2
    except ValueError as error:
        print(f"Could not fetch order: {error}")
await main()

{'order_id': 1, 'status': 'shipped'}
Could not fetch order: Order 2 was not found


When using `asyncio.gather()`, an exception can cause the awaited call to raise. For a beginner-friendly pattern, handle errors inside each task when you want to collect both successful results and readable error messages.

In [15]:
import asyncio


async def get_order(order_id):
    """Retrieve one order by ID."""

    # Single responsibility: retrieve an order. Keeping lookup logic here
    # means changes to how orders are fetched belong in this function.
    await asyncio.sleep(0.2)

    # Simulate the requested order not existing.
    if order_id == 2:
        raise ValueError("Order 2 was not found")

    return {"order_id": order_id, "status": "shipped"}


async def safe_get_order(order_id):
    """Retrieve an order and convert a lookup error into a result."""

    # Single responsibility: handle the expected lookup error and return it
    # in a predictable format. This keeps error handling separate from lookup.
    try:
        return await get_order(order_id)
    except ValueError as error:
        return {"order_id": order_id, "error": str(error)}


async def main():
    """Run several lookups concurrently and print their results."""

    # Single responsibility: coordinate the lookups and display their results.
    # gather runs them concurrently and returns results in call order.
    results = await asyncio.gather(
        safe_get_order(1),
        safe_get_order(2),
        safe_get_order(3),
    )

    for result in results:
        print(result)


# Start the asynchronous program.
await main()

{'order_id': 1, 'status': 'shipped'}
{'order_id': 2, 'error': 'Order 2 was not found'}
{'order_id': 3, 'status': 'shipped'}


### Add a timeout

`asyncio.wait_for()` stops waiting for an operation after the specified number of seconds. In real API calls, also configure the HTTP client's own timeout.

In [17]:
import asyncio


async def slow_operation():
    # Simulate work that takes 3 seconds to finish.
    await asyncio.sleep(3)

    # This value is returned only if the operation finishes.
    return "Finished"


async def main():
    try:
        # Run slow_operation(), but wait for at most 1 second.
        # Since the operation sleeps for 3 seconds, wait_for raises
        # asyncio.TimeoutError before it can return "Finished".
        result = await asyncio.wait_for(
            slow_operation(),
            timeout=1,
        )

        # This line runs only if the operation finishes within the timeout.
        print(result)

    except asyncio.TimeoutError:
        # Handle the timeout by displaying a message.
        print("The operation took too long.")

        # Raise a new timeout error so the caller also knows the operation
        # failed. If this is not caught elsewhere, the program ends with an error.
        raise TimeoutError


# Run the asynchronous main function.
await main()

The operation took too long.


TimeoutError: 

## 5. Real API calls with HTTPX (requires internet)

HTTPX provides `AsyncClient` for asynchronous HTTP requests. It is similar to the regular HTTP client introduced in Day 09, but its network calls are awaitable.

If HTTPX is not installed in your environment, run the next cell once.

In [ ]:
# Uncomment and run only if HTTPX is not installed:
# %pip install httpx

In [18]:
import asyncio
import httpx

# Base address for the DummyJSON API.
BASE_URL = "https://dummyjson.com"


async def fetch_product(client, product_id):
    # Request one product. The timeout limits how long this request can wait
    # for a response before HTTPX raises a timeout error.
    response = await client.get(
        f"{BASE_URL}/products/{product_id}",
        timeout=10.0,
    )

    # Raise an HTTPStatusError if the server responds with an unsuccessful
    # status code, such as 404. This prevents treating an error response
    # as if it were a successful product.
    response.raise_for_status()

    # Parse the response body from JSON into Python data
    # (for this API, a dictionary containing the product details).
    return response.json()


async def main():
    # Open one asynchronous HTTP client for all three requests.
    # The async-with block closes the client when the requests are done.
    async with httpx.AsyncClient() as client:

        # Start all three product requests concurrently.
        # gather waits for them to finish and returns their results
        # in the same order as the calls below.
        products = await asyncio.gather(
            fetch_product(client, 1),
            fetch_product(client, 2),
            fetch_product(client, 3),
        )

    # Print each product's ID and title.
    for product in products:
        print(product["id"], product["title"])


# Run this cell when you have an internet connection.
await main()

1 Essence Mascara Lash Princess
2 Eyeshadow Palette with Mirror
3 Powder Canister


### What happened?

1. We created one `AsyncClient` and reused it.
2. Each `client.get()` call is preceded by `await`.
3. `asyncio.gather()` runs the independent requests concurrently.
4. `raise_for_status()` makes HTTP errors visible instead of silently treating them as success.
5. The response body is JSON, which Python converts into dictionaries and lists.

In a notebook environment that already has an active event loop, use `await main()` instead of `asyncio.run(main())`.

## 6. Validate API data with Pydantic

An API response is external data. Before relying on it in application logic, we can validate its structure with Pydantic.

This section connects **Day 08 (Pydantic)** and **Day 09 (HTTP requests)** with async Python.

In [ ]:
# Uncomment and run only if Pydantic is not installed:
# %pip install pydantic

In [20]:
import asyncio
import httpx
from pydantic import BaseModel


# Base address for the DummyJSON API.
BASE_URL = "https://dummyjson.com"


class Product(BaseModel):
    # Define the fields and expected types for a product.
    # Pydantic checks incoming data against these declarations.
    id: int
    title: str
    price: float


async def fetch_validated_product(client, product_id):
    # Request one product from the API. The timeout limits how long
    # the request can wait for a response.
    response = await client.get(
        f"{BASE_URL}/products/{product_id}",
        timeout=10.0,
    )

    # Raise an error if the server returns an unsuccessful HTTP status code.
    response.raise_for_status()

    # Parse the response body as JSON, then validate its fields and types
    # against the Product model. Pydantic raises a ValidationError if the
    # returned data does not match the model.
    return Product.model_validate(response.json())


async def main():
    # Open one asynchronous HTTP client to reuse for all three requests.
    async with httpx.AsyncClient() as client:

        # Fetch all three products concurrently.
        # Results are returned in the same order as the calls.
        products = await asyncio.gather(
            fetch_validated_product(client, 1),
            fetch_validated_product(client, 2),
            fetch_validated_product(client, 3)
        )
        print("Output of Gather:",products)

    # Convert each validated Product model to a regular dictionary,
    # then print it.
    for product in products:
        print(product.model_dump())


# Run the asynchronous main function.
# In a regular Python script, asyncio.run() starts and manages the event loop.
await main()

Output of Gather: [Product(id=1, title='Essence Mascara Lash Princess', price=9.99), Product(id=2, title='Eyeshadow Palette with Mirror', price=19.99), Product(id=3, title='Powder Canister', price=14.99)]
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99}
{'id': 2, 'title': 'Eyeshadow Palette with Mirror', 'price': 19.99}
{'id': 3, 'title': 'Powder Canister', 'price': 14.99}


## 7. Limit how many requests run at once (optional)

If an application launches hundreds of requests at the same time, it may overwhelm the service or hit rate limits. An `asyncio.Semaphore` can limit how many operations are active simultaneously.

You only need to understand the idea for now: **a semaphore is a concurrency limit.**

In [31]:
import asyncio


async def limited_customer(customer_id, semaphore):
    # Acquire a semaphore slot before entering this section.
    # With a limit of 2, no more than two calls can be inside it at once.
    async with semaphore:
        print(f"Starting customer {customer_id}")

        # Simulate work that takes half a second.
        await asyncio.sleep(0.5)

        # Return a result for this customer.
        return f"Customer {customer_id}"


async def main():
    # Store the coroutine objects that will perform the customer work.
    task_list = []

    # Allow at most two coroutines to enter the protected section at a time.
    semaphore = asyncio.Semaphore(2)

    for customer_id in range(1, 6):
        # Calling an async function creates a coroutine object.
        # It does not create or start an asyncio Task by itself.
        customer_coroutine = limited_customer(customer_id, semaphore)
        task_list.append(customer_coroutine)

    # Show the list and its type. The list contains coroutine objects.
    print(task_list, type(task_list))

    # gather schedules the coroutines to run concurrently and waits for them.
    # The semaphore limits how many can run inside the protected section.
    # Results are returned in the same order as the coroutines in task_list.
    result = await asyncio.gather(*task_list)
    print(result)


# Run the asynchronous main function.
await main()

[<coroutine object limited_customer at 0x108800e50>, <coroutine object limited_customer at 0x1088dce50>, <coroutine object limited_customer at 0x1088dd300>, <coroutine object limited_customer at 0x1088dc4f0>, <coroutine object limited_customer at 0x1088dcc70>] <class 'list'>
Starting customer 1
Starting customer 2
Starting customer 3
Starting customer 4
Starting customer 5
['Customer 1', 'Customer 2', 'Customer 3', 'Customer 4', 'Customer 5']


## 8. How async fits into AI engineering

Imagine an AI assistant needs to gather customer details, order status, and product information before composing a response.

A simplified flow could be:

1. Call independent APIs concurrently with HTTPX.
2. Validate responses with Pydantic.
3. Combine the validated results.
4. Pass the relevant context to an LLM or agent.

Async helps reduce idle waiting when the work involves multiple independent I/O calls. It does **not** guarantee that the LLM itself will generate tokens faster.

In [34]:
import asyncio

# A small offline example: gather three pieces of context concurrently.

async def get_customer_context():
    # Simulate an asynchronous lookup that takes 0.3 seconds.
    await asyncio.sleep(0.3)

    # Return the customer information for the support context.
    return {"customer": "Maya"}


async def get_order_context():
    # Simulate an asynchronous lookup that takes 0.3 seconds.
    await asyncio.sleep(0.3)

    # Return the order information for the support context.
    return {"order_status": "shipped"}


async def get_product_context():
    # Simulate an asynchronous lookup that takes 0.3 seconds.
    await asyncio.sleep(0.3)

    # Return the product information for the support context.
    return {"product": "Wireless headphones"}


async def build_support_context():
    # Start all three lookups concurrently and wait for all of them to finish.
    # Each result is assigned to the matching variable in the same order
    # as the calls passed to gather.
    customer, order, product = await asyncio.gather(
        get_customer_context(),
        get_order_context(),
        get_product_context(),
    )

    # Combine the three dictionaries into one support context dictionary.
    # The ** syntax copies each dictionary's key-value pairs into the result.
    return {**customer, **order, **product}

# Run the function and return the combined context.
await build_support_context()

{'customer': 'Maya',
 'order_status': 'shipped',
 'product': 'Wireless headphones'}

## 9. Practice exercises

Try these before looking for help.

1. Change the sequential example to fetch five customers. What happens to the elapsed time?
2. Add a `get_invoice(invoice_id)` coroutine that waits for half a second and returns an invoice dictionary.
3. Use `asyncio.gather()` to fetch three invoices concurrently.
4. Make invoice 2 raise an exception. Handle the error so the other invoices still produce results.
5. Add a timeout to an operation that sleeps for two seconds.
6. In the HTTPX example, fetch products 4, 5, and 6, then validate each response with Pydantic.
7. **Challenge:** Use a semaphore to allow at most two simulated tasks to run inside the protected section at once.

### Quick knowledge check

- Does `async def` by itself make a function run concurrently? **No.**
- What does `await` do? **It waits for an awaitable result while allowing other async tasks to make progress.**
- When is async especially useful? **When the program spends time waiting for I/O.**
- Is async automatically faster for CPU-heavy calculations? **No.**
- Why reuse one HTTPX `AsyncClient`? **It allows connection reuse and avoids creating a new client for every request.**

## 10. Optional advanced topics — save for later

These are useful, but students do not need to master them in this introductory lesson:

- `asyncio.create_task()` — schedule a coroutine as a task.
- `asyncio.Queue` — coordinate producers and workers.
- `asyncio.to_thread()` — call blocking functions without blocking the event loop.
- Async database drivers such as `asyncpg` — database access using async I/O.
- Streaming LLM responses — process generated output as it arrives.

**Takeaway:** Learn `async def`, `await`, `asyncio.run()`, and `asyncio.gather()` first. Then practise asynchronous HTTP calls and response validation. Add queues, database pools, and more advanced concurrency patterns when building real applications.